# Task 1 — Tree-Based Optimization

This notebook starts directly from **Task 1** and is designed to run **without relying on earlier baseline cells**.
## Technique Reference Table

The table below gives a short plain-language description of each tree-based optimization technique and the foundational paper used as the reference point for the idea.

| Technique | Plain explanation | Foundational reference |
|---|---|---|
| Estimator Reduction | Use fewer trees or boosting rounds so the ensemble is smaller and cheaper to run. | Breiman, *Random Forests* (2001); Friedman, *Greedy Function Approximation: A Gradient Boosting Machine* (2001) |
| Depth Reduction | Limit tree depth so the model has fewer decision levels and less computation. | Breiman, Friedman, Olshen, Stone, *Classification and Regression Trees* (1984) |
| Leaf Reduction | Force the tree to end with fewer or larger leaves, which simplifies the structure. | Breiman et al., *Classification and Regression Trees* (1984) |
| Feature Subsampling | Use only a subset of features per split or per tree so the model is cheaper and often more regularized. | Breiman, *Random Forests* (2001) |
| Bin Reduction | Use fewer split bins in histogram-based boosting so training and memory usage are lower. | Ke et al., *LightGBM* (NeurIPS 2017) |

**Important note.** Not every technique is defined for every tree-based model.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
import json
import time
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

import xgboost as xgb
import lightgbm as lgb

sns.set_style("whitegrid")

## Task 1 Configuration

In [2]:
# Dataset paths
# Update these if your local paths are different.

CICEVSE_PATH = r'C:\Users\100987869\Desktop\EV-Survey-Models2\CICEVSE2024\Subsets\CICEVSE2024_12classes_kmeans_rule_sampled1000.csv'
CICIDS_PATH  = r'c:\Users\100987869\Downloads\cic_0.01km.csv'
NFTON_PATH   = r'C:\Users\100987869\Desktop\EV-Survey-Models2\NF-Ton-IoT-V2\NF-ToN-IoT-V2.parquet'

# Saved model registry paths
MODEL_DIR   = 'models'
MODELS_JSON = 'models.json'

TREE_MODEL_NAMES = [
    'Decision Tree',
    'Random Forest',
    'Extra Trees',
    'XGBoost',
    'LightGBM',
]

TEST_SIZE = 0.20
RANDOM_STATE = 42
NFTON_SUBSET_SIZE = 20000

print('Tree-based optimization configuration loaded.')

Tree-based optimization configuration loaded.


## Registry Helpers

In [3]:
def _load_registry() -> dict:
    if os.path.exists(MODELS_JSON):
        with open(MODELS_JSON, 'r') as f:
            return json.load(f)
    return {}

def _require_file(path_str: str, label: str):
    if not os.path.exists(path_str):
        raise FileNotFoundError(f'{label} not found: {path_str}')

def _normalise_path(path_str: str) -> str:
    return str(Path(path_str))

def _get_model_entry(dataset_tag: str, model_name: str):
    registry = _load_registry()
    exact_key = f'[{dataset_tag}] {model_name}'
    if exact_key in registry:
        return exact_key, registry[exact_key]

    normalized_exact = exact_key.strip().lower()
    for key, value in registry.items():
        if key.strip().lower() == normalized_exact:
            return key, value

    for key, value in registry.items():
        key_l = key.lower()
        if f'[{dataset_tag.lower()}]' in key_l and model_name.lower() in key_l:
            return key, value

    raise KeyError(f'Model entry not found in registry: {exact_key}')

def _load_saved_sklearn_model(dataset_tag: str, model_name: str):
    key, entry = _get_model_entry(dataset_tag, model_name)
    model_path = _normalise_path(entry['model_path'])
    _require_file(model_path, key)
    model = joblib.load(model_path)
    return model, key, entry

## Dataset Loading and Preprocessing

In [13]:
# CICEVSE2024
_require_file(CICEVSE_PATH, 'CICEVSE2024 dataset')
df_cicevse_2024 = pd.read_csv(CICEVSE_PATH)

labelencoder_cicevse = LabelEncoder()
df_cicevse_2024.iloc[:, -1] = labelencoder_cicevse.fit_transform(df_cicevse_2024.iloc[:, -1])

if df_cicevse_2024.isnull().values.any() or np.isinf(df_cicevse_2024.select_dtypes(include=[np.number])).values.any():
    df_cicevse_2024.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_cicevse_2024.fillna(0, inplace=True)

X_cicevse = df_cicevse_2024.drop(['Label'], axis=1)
y_cicevse = df_cicevse_2024.iloc[:, -1]

print('CICEVSE2024 shape:', X_cicevse.shape)
print('CICEVSE2024 classes:', len(np.unique(y_cicevse)))

# CICIDS2017
_require_file(CICIDS_PATH, 'CICIDS2017 dataset')
df_cicids = pd.read_csv(CICIDS_PATH)

labelencoder_cicids = LabelEncoder()
df_cicids.iloc[:, -1] = labelencoder_cicids.fit_transform(df_cicids.iloc[:, -1])

if df_cicids.isnull().values.any() or np.isinf(df_cicids.select_dtypes(include=[np.number])).values.any():
    df_cicids.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_cicids.fillna(0, inplace=True)

X_cicids = df_cicids.iloc[:, :-1]
y_cicids = df_cicids.iloc[:, -1]

print('CICIDS2017 shape:', X_cicids.shape)
print('CICIDS2017 classes:', len(np.unique(y_cicids)))

# NF-ToN-IoT-V2
_require_file(NFTON_PATH, 'NF-ToN-IoT-V2 dataset')
df_nfton = pd.read_parquet(NFTON_PATH)

label_col_candidates = ['Attack', 'Label', 'label']
label_col_nfton = None
for col in label_col_candidates:
    if col in df_nfton.columns:
        label_col_nfton = col
        break

if label_col_nfton is None:
    label_col_nfton = df_nfton.columns[-1]

df_nfton = df_nfton.copy()
for col in df_nfton.columns:
    if df_nfton[col].dtype == 'object':
        if col == label_col_nfton:
            continue
        df_nfton[col] = df_nfton[col].astype('category').cat.codes

labelencoder_nfton = LabelEncoder()
df_nfton[label_col_nfton] = labelencoder_nfton.fit_transform(df_nfton[label_col_nfton])

if df_nfton.isnull().values.any() or np.isinf(df_nfton.select_dtypes(include=[np.number])).values.any():
    df_nfton.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_nfton.fillna(0, inplace=True)

X_nfton = df_nfton.drop(columns=[label_col_nfton])
y_nfton = df_nfton[label_col_nfton]

if len(X_nfton) > NFTON_SUBSET_SIZE:
    X_nfton_sub, _, y_nfton_sub, _ = train_test_split(
        X_nfton, y_nfton,
        train_size=NFTON_SUBSET_SIZE,
        stratify=y_nfton,
        random_state=RANDOM_STATE
    )
else:
    X_nfton_sub, y_nfton_sub = X_nfton, y_nfton

print('NF-ToN-IoT-V2 original shape:', X_nfton.shape)
print('NF-ToN-IoT-V2 subset shape:', X_nfton_sub.shape)
print('NF-ToN-IoT-V2 classes:', len(np.unique(y_nfton_sub)))

CICEVSE2024 shape: (35941, 85)
CICEVSE2024 classes: 12
CICIDS2017 shape: (28303, 19)
CICIDS2017 classes: 2
NF-ToN-IoT-V2 original shape: (13135881, 42)
NF-ToN-IoT-V2 subset shape: (20000, 42)
NF-ToN-IoT-V2 classes: 10


## Fixed Hold-Out Splits

In [14]:
# Task uses fixed hold-out splits.
# Not finished due fold-specific baseline checkpoints and test splits not being available inside this standalone notebook.

X_train_cicevse, X_test_cicevse, y_train_cicevse, y_test_cicevse = train_test_split(
    X_cicevse, y_cicevse, test_size=TEST_SIZE, stratify=y_cicevse, random_state=RANDOM_STATE
)

X_train_cicids, X_test_cicids, y_train_cicids, y_test_cicids = train_test_split(
    X_cicids, y_cicids, test_size=TEST_SIZE, stratify=y_cicids, random_state=RANDOM_STATE
)

X_train_nfton, X_test_nfton, y_train_nfton, y_test_nfton = train_test_split(
    X_nfton_sub, y_nfton_sub, test_size=TEST_SIZE, stratify=y_nfton_sub, random_state=RANDOM_STATE
)

print('Hold-out splits prepared.')

Hold-out splits prepared.


## Tree-Based Model Loading

In [15]:
# CICEVSE2024 tree-based models
dt_cicevse_model, dt_cicevse_key, dt_cicevse_entry = _load_saved_sklearn_model('CICEVSE', 'Decision Tree')
rf_cicevse_model, rf_cicevse_key, rf_cicevse_entry = _load_saved_sklearn_model('CICEVSE', 'Random Forest')
et_cicevse_model, et_cicevse_key, et_cicevse_entry = _load_saved_sklearn_model('CICEVSE', 'Extra Trees')
#xgb_cicevse_model, xgb_cicevse_key, xgb_cicevse_entry = _load_saved_sklearn_model('CICEVSE', 'XGBoost')
lgb_cicevse_model, lgb_cicevse_key, lgb_cicevse_entry = _load_saved_sklearn_model('CICEVSE', 'LightGBM')
print('Loaded CICEVSE2024 tree-based models.')

# CICIDS2017 tree-based models
dt_cicids_model, dt_cicids_key, dt_cicids_entry = _load_saved_sklearn_model('CICIDS', 'Decision Tree')
rf_cicids_model, rf_cicids_key, rf_cicids_entry = _load_saved_sklearn_model('CICIDS', 'Random Forest')
et_cicids_model, et_cicids_key, et_cicids_entry = _load_saved_sklearn_model('CICIDS', 'Extra Trees')
#xgb_cicids_model, xgb_cicids_key, xgb_cicids_entry = _load_saved_sklearn_model('CICIDS', 'XGBoost')
lgb_cicids_model, lgb_cicids_key, lgb_cicids_entry = _load_saved_sklearn_model('CICIDS', 'LightGBM')
print('Loaded CICIDS2017 tree-based models.')

# NF-ToN-IoT-V2 tree-based models
dt_nfton_model, dt_nfton_key, dt_nfton_entry = _load_saved_sklearn_model('NFToN', 'Decision Tree')
rf_nfton_model, rf_nfton_key, rf_nfton_entry = _load_saved_sklearn_model('NFToN', 'Random Forest')
et_nfton_model, et_nfton_key, et_nfton_entry = _load_saved_sklearn_model('NFToN', 'Extra Trees')
#xgb_nfton_model, xgb_nfton_key, xgb_nfton_entry = _load_saved_sklearn_model('NFToN', 'XGBoost')
lgb_nfton_model, lgb_nfton_key, lgb_nfton_entry = _load_saved_sklearn_model('NFToN', 'LightGBM')
print('Loaded NF-ToN-IoT-V2 tree-based models.')

Loaded CICEVSE2024 tree-based models.
Loaded CICIDS2017 tree-based models.
Loaded NF-ToN-IoT-V2 tree-based models.


## Optimization Helpers

In [16]:
def get_model_family(model_name: str) -> str:
    tree_names = ['Decision Tree', 'Random Forest', 'Extra Trees', 'XGBoost', 'LightGBM']
    return 'Tree-Based' if model_name in tree_names else 'Classical'

def compute_macro_f1(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    f1_vals = []
    for cls_name, cls_metrics in report.items():
        if cls_name in ['accuracy', 'macro avg', 'weighted avg']:
            continue
        if isinstance(cls_metrics, dict) and 'f1-score' in cls_metrics:
            f1_vals.append(cls_metrics['f1-score'])
    return float(np.mean(f1_vals)) if len(f1_vals) > 0 else np.nan

def get_pickle_model_size_bytes(model_obj):
    with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as tmp_file:
        joblib.dump(model_obj, tmp_file.name)
        tmp_path = tmp_file.name
    size_bytes = os.path.getsize(tmp_path)
    os.remove(tmp_path)
    return size_bytes

def evaluate_sklearn_model(model_obj, X_test, y_test, model_name: str, model_size_bytes: int):
    start = time.time()
    y_pred = model_obj.predict(X_test)
    infer_time_s = time.time() - start

    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)

    return {
        'Model': model_name,
        'Family': get_model_family(model_name),
        'Accuracy (%)': accuracy_score(y_test, y_pred) * 100,
        'Weighted F1 (%)': f * 100,
        'Macro F1 (%)': compute_macro_f1(y_test, y_pred) * 100,
        'Precision (%)': p * 100,
        'Recall (%)': r * 100,
        'Inference Time (ms/sample)': (infer_time_s / len(X_test)) * 1000,
        'Model Size (bytes)': model_size_bytes,
    }

def make_result_row(dataset_name, technique_name, baseline_name, optimized_name, baseline_metrics, optimized_metrics, note=''):
    return {
        'Dataset': dataset_name,
        'Technique': technique_name,
        'Baseline Model': baseline_name,
        'Optimized Model': optimized_name,
        'Baseline Accuracy (%)': baseline_metrics['Accuracy (%)'],
        'Optimized Accuracy (%)': optimized_metrics['Accuracy (%)'],
        'Baseline Weighted F1 (%)': baseline_metrics['Weighted F1 (%)'],
        'Optimized Weighted F1 (%)': optimized_metrics['Weighted F1 (%)'],
        'Baseline Macro F1 (%)': baseline_metrics['Macro F1 (%)'],
        'Optimized Macro F1 (%)': optimized_metrics['Macro F1 (%)'],
        'Baseline Inference Time (ms/sample)': baseline_metrics['Inference Time (ms/sample)'],
        'Optimized Inference Time (ms/sample)': optimized_metrics['Inference Time (ms/sample)'],
        'Baseline Model Size (bytes)': baseline_metrics['Model Size (bytes)'],
        'Optimized Model Size (bytes)': optimized_metrics['Model Size (bytes)'],
        'Weighted F1 Change (%)': optimized_metrics['Weighted F1 (%)'] - baseline_metrics['Weighted F1 (%)'],
        'Macro F1 Change (%)': optimized_metrics['Macro F1 (%)'] - baseline_metrics['Macro F1 (%)'],
        'Inference Time Change (%)': ((optimized_metrics['Inference Time (ms/sample)'] - baseline_metrics['Inference Time (ms/sample)']) / max(baseline_metrics['Inference Time (ms/sample)'], 1e-9)) * 100,
        'Model Size Change (%)': ((optimized_metrics['Model Size (bytes)'] - baseline_metrics['Model Size (bytes)']) / max(baseline_metrics['Model Size (bytes)'], 1e-9)) * 100,
        'Note': note,
    }

def _plot_change_bars(result_df, value_col, title_text):
    plot_df = result_df.copy().sort_values(value_col, ascending=True)
    fig, ax = plt.subplots(figsize=(14, max(6, len(plot_df) * 0.35)))
    bars = ax.barh(plot_df['Optimized Model'], plot_df[value_col], color=sns.color_palette('muted', len(plot_df)))
    for bar, val in zip(bars, plot_df[value_col]):
        if pd.isna(val):
            continue
        x_pos = val + 0.05 if val >= 0 else val - 0.8
        ax.text(x_pos, bar.get_y() + bar.get_height() / 2, f'{val:.2f}%', va='center', fontsize=8)
    ax.set_xlabel(value_col)
    ax.set_ylabel('Optimized Model')
    ax.set_title(title_text)
    ax.grid(True, axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

def _plot_baseline_vs_optimized(result_df, metric_baseline_col, metric_optimized_col, title_text):
    labels = result_df['Optimized Model'].tolist()
    baseline_vals = result_df[metric_baseline_col].tolist()
    optimized_vals = result_df[metric_optimized_col].tolist()
    y = np.arange(len(labels))
    height = 0.35
    fig, ax = plt.subplots(figsize=(14, max(6, len(labels) * 0.35)))
    bars1 = ax.barh(y - height / 2, baseline_vals, height, label='Baseline', color=sns.color_palette('deep')[0])
    bars2 = ax.barh(y + height / 2, optimized_vals, height, label='Optimized', color=sns.color_palette('deep')[1])
    for bar, val in zip(bars1, baseline_vals):
        if pd.isna(val):
            continue
        ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=8)
    for bar, val in zip(bars2, optimized_vals):
        if pd.isna(val):
            continue
        ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=8)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel('Value')
    ax.set_ylabel('Model')
    ax.set_title(title_text)
    ax.grid(True, axis='x', linestyle='--', alpha=0.5)
    ax.legend()
    plt.tight_layout()
    plt.show()

def _plot_cost_vs_f1(result_df, title_text):
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.scatterplot(data=result_df, x='Optimized Inference Time (ms/sample)', y='Optimized Weighted F1 (%)', hue='Technique', s=120, ax=ax)
    for _, row in result_df.iterrows():
        if pd.isna(row['Optimized Inference Time (ms/sample)']) or pd.isna(row['Optimized Weighted F1 (%)']):
            continue
        ax.text(row['Optimized Inference Time (ms/sample)'], row['Optimized Weighted F1 (%)'], row['Optimized Model'], fontsize=7, ha='left', va='bottom')
    ax.set_xlabel('Optimized Inference Time (ms/sample)')
    ax.set_ylabel('Optimized Weighted F1 (%)')
    ax.set_title(title_text)
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

In [17]:
def train_tree_variant(model_name, baseline_model, X_train, y_train, technique_name):
    lower_name = model_name.lower()

    if technique_name == 'Estimator Reduction':
        if 'random forest' in lower_name or 'extra trees' in lower_name:
            n_est = baseline_model.get_params().get('n_estimators', 100)
            variant = clone(baseline_model).set_params(n_estimators=max(10, int(n_est * 0.5)))
        elif 'xgboost' in lower_name or 'lightgbm' in lower_name:
            n_est = baseline_model.get_params().get('n_estimators', 100)
            variant = clone(baseline_model).set_params(n_estimators=max(20, int(n_est * 0.5)))
        elif 'decision tree' in lower_name:
            return None, 'Not finished due estimator reduction not being applicable to a single Decision Tree.'
        else:
            return None, 'Not finished due unsupported tree model.'
    elif technique_name == 'Depth Reduction':
        base_depth = baseline_model.get_params().get('max_depth', None)
        if base_depth in [None, -1]:
            new_depth = 8
        else:
            new_depth = max(2, int(max(base_depth, 4) * 0.7))
        variant = clone(baseline_model).set_params(max_depth=new_depth)
    elif technique_name == 'Leaf Reduction':
        if 'decision tree' in lower_name or 'random forest' in lower_name or 'extra trees' in lower_name:
            params = {'min_samples_leaf': max(2, baseline_model.get_params().get('min_samples_leaf', 1) + 1)}
            if 'max_leaf_nodes' in baseline_model.get_params():
                base_leaf = baseline_model.get_params().get('max_leaf_nodes', None)
                params['max_leaf_nodes'] = 32 if base_leaf is None else max(8, int(max(base_leaf, 16) * 0.7))
            variant = clone(baseline_model).set_params(**params)
        elif 'lightgbm' in lower_name:
            base_leaves = baseline_model.get_params().get('num_leaves', 31)
            min_child = baseline_model.get_params().get('min_child_samples', 20)
            variant = clone(baseline_model).set_params(num_leaves=max(8, int(base_leaves * 0.7)), min_child_samples=max(20, min_child + 10))
        elif 'xgboost' in lower_name:
            base_weight = baseline_model.get_params().get('min_child_weight', 1)
            variant = clone(baseline_model).set_params(min_child_weight=max(2, base_weight + 1))
        else:
            return None, 'Not finished due unsupported tree model.'
    elif technique_name == 'Feature Subsampling':
        if 'decision tree' in lower_name or 'random forest' in lower_name or 'extra trees' in lower_name:
            variant = clone(baseline_model).set_params(max_features='sqrt')
        elif 'xgboost' in lower_name:
            variant = clone(baseline_model).set_params(colsample_bytree=0.7, colsample_bylevel=0.7)
        elif 'lightgbm' in lower_name:
            variant = clone(baseline_model).set_params(colsample_bytree=0.7)
        else:
            return None, 'Not finished due unsupported tree model.'
    elif technique_name == 'Bin Reduction':
        if 'lightgbm' in lower_name:
            variant = clone(baseline_model).set_params(max_bin=63)
        elif 'xgboost' in lower_name and 'max_bin' in baseline_model.get_params():
            variant = clone(baseline_model).set_params(max_bin=64, tree_method='hist')
        else:
            return None, 'Not finished due bin reduction not being defined for this tree model in the current implementation.'
    else:
        return None, 'Not finished due unknown technique.'

    variant.fit(X_train, y_train)
    return variant, ''

## Baseline Metrics

In [18]:
# CICEVSE2024

baseline_rows_cicevse = []

for model_name, model_obj in [
    ('Decision Tree', dt_cicevse_model),
    ('Random Forest', rf_cicevse_model),
    ('Extra Trees', et_cicevse_model),
    #('XGBoost', xgb_cicevse_model),
    ('LightGBM', lgb_cicevse_model)
]:
    model_size_bytes = get_pickle_model_size_bytes(model_obj)
    metrics = evaluate_sklearn_model(model_obj, X_test_cicevse, y_test_cicevse, model_name, model_size_bytes=model_size_bytes)
    baseline_rows_cicevse.append(metrics)

baseline_df_cicevse = pd.DataFrame(baseline_rows_cicevse)
display(baseline_df_cicevse)

,Model,Family,Accuracy (%),Weighted F1 (%),Macro F1 (%),Precision (%),Recall (%),Inference Time (ms/sample),Model Size (bytes)
0,Decision Tree,Tree-Based,100.000000,100.000000,100.000000,100.000000,100.000000,0.000905,21209
1,Random Forest,Tree-Based,100.000000,100.000000,100.000000,100.000000,100.000000,0.006804,8990265
2,Extra Trees,Tree-Based,100.000000,100.000000,100.000000,100.000000,100.000000,0.009510,27623641
3,LightGBM,Tree-Based,97.176241,97.182878,96.909617,97.597134,97.176241,0.204402,7753396


In [19]:
# CICIDS2017

baseline_rows_cicids = []

for model_name, model_obj in [
    ('Decision Tree', dt_cicids_model),
    ('Random Forest', rf_cicids_model),
    ('Extra Trees', et_cicids_model),
    #('XGBoost', xgb_cicids_model),
    ('LightGBM', lgb_cicids_model)
]:
    model_size_bytes = get_pickle_model_size_bytes(model_obj)
    metrics = evaluate_sklearn_model(model_obj, X_test_cicids, y_test_cicids, model_name, model_size_bytes=model_size_bytes)
    baseline_rows_cicids.append(metrics)

baseline_df_cicids = pd.DataFrame(baseline_rows_cicids)
display(baseline_df_cicids)

,Model,Family,Accuracy (%),Weighted F1 (%),Macro F1 (%),Precision (%),Recall (%),Inference Time (ms/sample),Model Size (bytes)
0,Decision Tree,Tree-Based,70.464582,70.364998,53.410838,70.266852,70.464582,0.000351,28761
1,Random Forest,Tree-Based,79.667903,74.695820,54.074696,74.544116,79.667903,0.003267,3327497
2,Extra Trees,Tree-Based,79.826886,71.091409,44.390963,64.079205,79.826886,0.002647,12660569
3,LightGBM,Tree-Based,80.074192,71.213716,44.467334,64.118762,80.074192,0.000773,713908


In [23]:
# NF-ToN-IoT-V2

baseline_rows_nfton = []

for model_name, model_obj in [
    ('Decision Tree', dt_nfton_model),
    ('Random Forest', rf_nfton_model),
    ('Extra Trees', et_nfton_model),
    #('XGBoost', xgb_nfton_model),
    ('LightGBM', lgb_nfton_model)
]:
    model_size_bytes = get_pickle_model_size_bytes(model_obj)
    metrics = evaluate_sklearn_model(model_obj, X_test_nfton.drop(columns=['Attack'], errors='ignore'), y_test_nfton, model_name, model_size_bytes=model_size_bytes)
    baseline_rows_nfton.append(metrics)

baseline_df_nfton = pd.DataFrame(baseline_rows_nfton)
display(baseline_df_nfton)

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Label


## Estimator Reduction

In [ ]:
# Estimator Reduction — CICEVSE2024

estimator_reduction_rows_cicevse = []

for model_name, baseline_model in [
    ('Decision Tree', dt_cicevse_model),
    ('Random Forest', rf_cicevse_model),
    ('Extra Trees', et_cicevse_model),
    #('XGBoost', xgb_cicevse_model),
    ('LightGBM', lgb_cicevse_model)
]:
    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_cicevse, y_train_cicevse, 'Estimator Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Estimator Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicevse, y_test_cicevse, f'{model_name} [Estimator Reduction]', model_size_bytes=variant_size_bytes)

    estimator_reduction_rows_cicevse.append(
        make_result_row('CICEVSE2024', 'Estimator Reduction', model_name, f'{model_name} [Estimator Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

estimator_reduction_df_cicevse = pd.DataFrame(estimator_reduction_rows_cicevse)
display(estimator_reduction_df_cicevse)

In [ ]:
# Estimator Reduction — CICIDS2017

estimator_reduction_rows_cicids = []

for model_name, baseline_model in [
    ('Decision Tree', dt_cicids_model),
    ('Random Forest', rf_cicids_model),
    ('Extra Trees', et_cicids_model),
    #('XGBoost', xgb_cicids_model),
    ('LightGBM', lgb_cicids_model)
]:
    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_cicids, y_train_cicids, 'Estimator Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Estimator Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicids, y_test_cicids, f'{model_name} [Estimator Reduction]', model_size_bytes=variant_size_bytes)

    estimator_reduction_rows_cicids.append(
        make_result_row('CICIDS2017', 'Estimator Reduction', model_name, f'{model_name} [Estimator Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

estimator_reduction_df_cicids = pd.DataFrame(estimator_reduction_rows_cicids)
display(estimator_reduction_df_cicids)

In [ ]:
# Estimator Reduction — NF-ToN-IoT-V2

estimator_reduction_rows_nfton = []

for model_name, baseline_model in [
    ('Decision Tree', dt_nfton_model),
    ('Random Forest', rf_nfton_model),
    ('Extra Trees', et_nfton_model),
    #('XGBoost', xgb_nfton_model),
    ('LightGBM', lgb_nfton_model)
]:
    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_nfton, y_train_nfton, 'Estimator Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Estimator Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_nfton, y_test_nfton, f'{model_name} [Estimator Reduction]', model_size_bytes=variant_size_bytes)

    estimator_reduction_rows_nfton.append(
        make_result_row('NF-ToN-IoT-V2', 'Estimator Reduction', model_name, f'{model_name} [Estimator Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

estimator_reduction_df_nfton = pd.DataFrame(estimator_reduction_rows_nfton)
display(estimator_reduction_df_nfton)

## Depth Reduction

In [ ]:
# Depth Reduction — CICEVSE2024

depth_reduction_rows_cicevse = []

for model_name, baseline_model in [
    ('Decision Tree', dt_cicevse_model),
    ('Random Forest', rf_cicevse_model),
    ('Extra Trees', et_cicevse_model),
    #('XGBoost', xgb_cicevse_model),
    ('LightGBM', lgb_cicevse_model)
]:
    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_cicevse, y_train_cicevse, 'Depth Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Depth Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicevse, y_test_cicevse, f'{model_name} [Depth Reduction]', model_size_bytes=variant_size_bytes)

    depth_reduction_rows_cicevse.append(
        make_result_row('CICEVSE2024', 'Depth Reduction', model_name, f'{model_name} [Depth Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

depth_reduction_df_cicevse = pd.DataFrame(depth_reduction_rows_cicevse)
display(depth_reduction_df_cicevse)

In [ ]:
# Depth Reduction — CICIDS2017

depth_reduction_rows_cicids = []

for model_name, baseline_model in [
    ('Decision Tree', dt_cicids_model),
    ('Random Forest', rf_cicids_model),
    ('Extra Trees', et_cicids_model),
    #('XGBoost', xgb_cicids_model),
    ('LightGBM', lgb_cicids_model)
]:
    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_cicids, y_train_cicids, 'Depth Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Depth Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicids, y_test_cicids, f'{model_name} [Depth Reduction]', model_size_bytes=variant_size_bytes)

    depth_reduction_rows_cicids.append(
        make_result_row('CICIDS2017', 'Depth Reduction', model_name, f'{model_name} [Depth Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

depth_reduction_df_cicids = pd.DataFrame(depth_reduction_rows_cicids)
display(depth_reduction_df_cicids)

In [ ]:
# Depth Reduction — NF-ToN-IoT-V2

depth_reduction_rows_nfton = []

for model_name, baseline_model in [
    ('Decision Tree', dt_nfton_model),
    ('Random Forest', rf_nfton_model),
    ('Extra Trees', et_nfton_model),
    #('XGBoost', xgb_nfton_model),
    ('LightGBM', lgb_nfton_model)
]:
    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_nfton, y_train_nfton, 'Depth Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Depth Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_nfton, y_test_nfton, f'{model_name} [Depth Reduction]', model_size_bytes=variant_size_bytes)

    depth_reduction_rows_nfton.append(
        make_result_row('NF-ToN-IoT-V2', 'Depth Reduction', model_name, f'{model_name} [Depth Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

depth_reduction_df_nfton = pd.DataFrame(depth_reduction_rows_nfton)
display(depth_reduction_df_nfton)

## Leaf Reduction

In [ ]:
# Leaf Reduction — CICEVSE2024

leaf_reduction_rows_cicevse = []

for model_name, baseline_model in [
    ('Decision Tree', dt_cicevse_model),
    ('Random Forest', rf_cicevse_model),
    ('Extra Trees', et_cicevse_model),
    #('XGBoost', xgb_cicevse_model),
    ('LightGBM', lgb_cicevse_model)
]:
    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_cicevse, y_train_cicevse, 'Leaf Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Leaf Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicevse, y_test_cicevse, f'{model_name} [Leaf Reduction]', model_size_bytes=variant_size_bytes)

    leaf_reduction_rows_cicevse.append(
        make_result_row('CICEVSE2024', 'Leaf Reduction', model_name, f'{model_name} [Leaf Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

leaf_reduction_df_cicevse = pd.DataFrame(leaf_reduction_rows_cicevse)
display(leaf_reduction_df_cicevse)

In [ ]:
# Leaf Reduction — CICIDS2017

leaf_reduction_rows_cicids = []

for model_name, baseline_model in [
    ('Decision Tree', dt_cicids_model),
    ('Random Forest', rf_cicids_model),
    ('Extra Trees', et_cicids_model),
    #('XGBoost', xgb_cicids_model),
    ('LightGBM', lgb_cicids_model)
]:
    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_cicids, y_train_cicids, 'Leaf Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Leaf Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicids, y_test_cicids, f'{model_name} [Leaf Reduction]', model_size_bytes=variant_size_bytes)

    leaf_reduction_rows_cicids.append(
        make_result_row('CICIDS2017', 'Leaf Reduction', model_name, f'{model_name} [Leaf Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

leaf_reduction_df_cicids = pd.DataFrame(leaf_reduction_rows_cicids)
display(leaf_reduction_df_cicids)

In [ ]:
# Leaf Reduction — NF-ToN-IoT-V2

leaf_reduction_rows_nfton = []

for model_name, baseline_model in [
    ('Decision Tree', dt_nfton_model),
    ('Random Forest', rf_nfton_model),
    ('Extra Trees', et_nfton_model),
    #('XGBoost', xgb_nfton_model),
    ('LightGBM', lgb_nfton_model)
]:
    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_nfton, y_train_nfton, 'Leaf Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Leaf Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_nfton, y_test_nfton, f'{model_name} [Leaf Reduction]', model_size_bytes=variant_size_bytes)

    leaf_reduction_rows_nfton.append(
        make_result_row('NF-ToN-IoT-V2', 'Leaf Reduction', model_name, f'{model_name} [Leaf Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

leaf_reduction_df_nfton = pd.DataFrame(leaf_reduction_rows_nfton)
display(leaf_reduction_df_nfton)

## Feature Subsampling

In [ ]:
# Feature Subsampling — CICEVSE2024

feature_subsampling_rows_cicevse = []

for model_name, baseline_model in [
    ('Decision Tree', dt_cicevse_model),
    ('Random Forest', rf_cicevse_model),
    ('Extra Trees', et_cicevse_model),
    #('XGBoost', xgb_cicevse_model),
    ('LightGBM', lgb_cicevse_model)
]:
    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_cicevse, y_train_cicevse, 'Feature Subsampling')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Feature Subsampling]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicevse, y_test_cicevse, f'{model_name} [Feature Subsampling]', model_size_bytes=variant_size_bytes)

    feature_subsampling_rows_cicevse.append(
        make_result_row('CICEVSE2024', 'Feature Subsampling', model_name, f'{model_name} [Feature Subsampling]', baseline_metrics, optimized_metrics, note=note)
    )

feature_subsampling_df_cicevse = pd.DataFrame(feature_subsampling_rows_cicevse)
display(feature_subsampling_df_cicevse)

In [ ]:
# Feature Subsampling — CICIDS2017

feature_subsampling_rows_cicids = []

for model_name, baseline_model in [
    ('Decision Tree', dt_cicids_model),
    ('Random Forest', rf_cicids_model),
    ('Extra Trees', et_cicids_model),
    #('XGBoost', xgb_cicids_model),
    ('LightGBM', lgb_cicids_model)
]:
    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_cicids, y_train_cicids, 'Feature Subsampling')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Feature Subsampling]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicids, y_test_cicids, f'{model_name} [Feature Subsampling]', model_size_bytes=variant_size_bytes)

    feature_subsampling_rows_cicids.append(
        make_result_row('CICIDS2017', 'Feature Subsampling', model_name, f'{model_name} [Feature Subsampling]', baseline_metrics, optimized_metrics, note=note)
    )

feature_subsampling_df_cicids = pd.DataFrame(feature_subsampling_rows_cicids)
display(feature_subsampling_df_cicids)

In [ ]:
# Feature Subsampling — NF-ToN-IoT-V2

feature_subsampling_rows_nfton = []

for model_name, baseline_model in [
    ('Decision Tree', dt_nfton_model),
    ('Random Forest', rf_nfton_model),
    ('Extra Trees', et_nfton_model),
    #('XGBoost', xgb_nfton_model),
    ('LightGBM', lgb_nfton_model)
]:
    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_nfton, y_train_nfton, 'Feature Subsampling')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Feature Subsampling]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_nfton, y_test_nfton, f'{model_name} [Feature Subsampling]', model_size_bytes=variant_size_bytes)

    feature_subsampling_rows_nfton.append(
        make_result_row('NF-ToN-IoT-V2', 'Feature Subsampling', model_name, f'{model_name} [Feature Subsampling]', baseline_metrics, optimized_metrics, note=note)
    )

feature_subsampling_df_nfton = pd.DataFrame(feature_subsampling_rows_nfton)
display(feature_subsampling_df_nfton)

## Bin Reduction

In [ ]:
# Bin Reduction — CICEVSE2024

bin_reduction_rows_cicevse = []

for model_name, baseline_model in [
    ('Decision Tree', dt_cicevse_model),
    ('Random Forest', rf_cicevse_model),
    ('Extra Trees', et_cicevse_model),
    #('XGBoost', xgb_cicevse_model),
    ('LightGBM', lgb_cicevse_model)
]:
    baseline_metrics = baseline_df_cicevse[baseline_df_cicevse['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_cicevse, y_train_cicevse, 'Bin Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Bin Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicevse, y_test_cicevse, f'{model_name} [Bin Reduction]', model_size_bytes=variant_size_bytes)

    bin_reduction_rows_cicevse.append(
        make_result_row('CICEVSE2024', 'Bin Reduction', model_name, f'{model_name} [Bin Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

bin_reduction_df_cicevse = pd.DataFrame(bin_reduction_rows_cicevse)
display(bin_reduction_df_cicevse)

In [ ]:
# Bin Reduction — CICIDS2017

bin_reduction_rows_cicids = []

for model_name, baseline_model in [
    ('Decision Tree', dt_cicids_model),
    ('Random Forest', rf_cicids_model),
    ('Extra Trees', et_cicids_model),
    #('XGBoost', xgb_cicids_model),
    ('LightGBM', lgb_cicids_model)
]:
    baseline_metrics = baseline_df_cicids[baseline_df_cicids['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_cicids, y_train_cicids, 'Bin Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Bin Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_cicids, y_test_cicids, f'{model_name} [Bin Reduction]', model_size_bytes=variant_size_bytes)

    bin_reduction_rows_cicids.append(
        make_result_row('CICIDS2017', 'Bin Reduction', model_name, f'{model_name} [Bin Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

bin_reduction_df_cicids = pd.DataFrame(bin_reduction_rows_cicids)
display(bin_reduction_df_cicids)

In [ ]:
# Bin Reduction — NF-ToN-IoT-V2

bin_reduction_rows_nfton = []

for model_name, baseline_model in [
    ('Decision Tree', dt_nfton_model),
    ('Random Forest', rf_nfton_model),
    ('Extra Trees', et_nfton_model),
    #('XGBoost', xgb_nfton_model),
    ('LightGBM', lgb_nfton_model)
]:
    baseline_metrics = baseline_df_nfton[baseline_df_nfton['Model'] == model_name].iloc[0].to_dict()
    variant_model, note = train_tree_variant(model_name, baseline_model, X_train_nfton, y_train_nfton, 'Bin Reduction')

    if variant_model is None:
        optimized_metrics = {
            'Model': f'{model_name} [Bin Reduction]',
            'Family': 'Tree-Based',
            'Accuracy (%)': np.nan,
            'Weighted F1 (%)': np.nan,
            'Macro F1 (%)': np.nan,
            'Precision (%)': np.nan,
            'Recall (%)': np.nan,
            'Inference Time (ms/sample)': np.nan,
            'Model Size (bytes)': np.nan,
        }
    else:
        variant_size_bytes = get_pickle_model_size_bytes(variant_model)
        optimized_metrics = evaluate_sklearn_model(variant_model, X_test_nfton, y_test_nfton, f'{model_name} [Bin Reduction]', model_size_bytes=variant_size_bytes)

    bin_reduction_rows_nfton.append(
        make_result_row('NF-ToN-IoT-V2', 'Bin Reduction', model_name, f'{model_name} [Bin Reduction]', baseline_metrics, optimized_metrics, note=note)
    )

bin_reduction_df_nfton = pd.DataFrame(bin_reduction_rows_nfton)
display(bin_reduction_df_nfton)

## Combined Task 1 Result Table

In [ ]:
tree_all_df = pd.concat([
    estimator_reduction_df_cicevse, estimator_reduction_df_cicids, estimator_reduction_df_nfton,
    depth_reduction_df_cicevse, depth_reduction_df_cicids, depth_reduction_df_nfton,
    leaf_reduction_df_cicevse, leaf_reduction_df_cicids, leaf_reduction_df_nfton,
    feature_subsampling_df_cicevse, feature_subsampling_df_cicids, feature_subsampling_df_nfton,
    bin_reduction_df_cicevse, bin_reduction_df_cicids, bin_reduction_df_nfton,
], ignore_index=True)

display(tree_all_df)

## Final Task 1 Plots

In [ ]:
_plot_change_bars(tree_all_df, 'Weighted F1 Change (%)', 'Tree-Based Optimization — Weighted F1 Change')
_plot_change_bars(tree_all_df, 'Macro F1 Change (%)', 'Tree-Based Optimization — Macro F1 Change')
_plot_change_bars(tree_all_df, 'Inference Time Change (%)', 'Tree-Based Optimization — Inference Time Change')
_plot_change_bars(tree_all_df, 'Model Size Change (%)', 'Tree-Based Optimization — Model Size Change')

_plot_baseline_vs_optimized(tree_all_df, 'Baseline Weighted F1 (%)', 'Optimized Weighted F1 (%)', 'Tree-Based Optimization — Baseline vs Optimized Weighted F1')
_plot_baseline_vs_optimized(tree_all_df, 'Baseline Inference Time (ms/sample)', 'Optimized Inference Time (ms/sample)', 'Tree-Based Optimization — Baseline vs Optimized Inference Time')
_plot_cost_vs_f1(tree_all_df, 'Tree-Based Optimization — Optimized Weighted F1 vs Inference Time')